# AML-GNN: notes on building and improving our own GNN

These notes cover how `amlgnn` was built, why each choice was made, how it compares with the
GraphSAGE and QGNN models already in the ML service, and what to try next.

**Files**

| File | What it is |
|---|---|
| `amlgnn.py` | Features, model and scoring function. Used by both training and the ML service, so the two can't drift apart. |
| `train_amlgnn.py` | Downloads the data (kagglehub), builds batches, trains, compares all models, saves the bundle. |
| `model/amlgnn_v1.pt`, `model/amlgnn_v2.pt` | Trained bundles: weights, feature scaling, threshold, config and test results. |
| `service.py` | Serves it at `POST /models/amlgnn/predict`, same request and response as the other models. |
| `data/` | Cached training batches and graphs (git-ignored, rebuilt by the training script). |

**Run it**

```bash
cd ML
uv run python train_amlgnn.py              # train v2 + compare (first run downloads ~3 GB)
uv run python train_amlgnn.py --with-qgnn  # also compare QGNN (+~10 min)
uv run main.py                             # ML service on :8001
```

Backend: set `ML_MODEL=amlgnn` in `Backend/.env` to score batches with it.


## 1. Starting point: `aml-graph-gnn-detection.ipynb`

The notebook trains GCN, GAT and GraphSAGE edge classifiers on IBM **HI-Small**:

- 200k transactions sampled **at random** from 5M, time-ordered 70/15/15 split
- 8 node features (log totals, counts, unique counterparties, averages per account)
- 9 edge features (log amounts, amount ratio, cross-currency, hour sin/cos, weekend, night,
  payment format as one *label-encoded number*)
- 2 graph layers -> `[sender, receiver, edge features]` -> MLP

Its best test PR-AUC (AUPRC) was **0.038** (GAT), at a 0.10% laundering rate.

**What held it back**

1. **Random sampling destroys the graph.** 200k random edges over 180k accounts is about one
   transaction per account, so there are almost no neighbourhoods to learn from.
2. **The graph layers ignore transaction features and direction.** GCN/GAT/SAGE only pass
   account features; amount, currency and format only reach the final MLP. Sending and
   receiving are mixed together.
3. **Payment format as a single number** (0-6) implies an order that doesn't exist.
4. **One graph for train/val/test**, while our service scores separate 50k-row batches.

We kept its overall design (node aggregates + GNN + edge MLP, weighted BCE, early stopping on
validation AUPRC) and fixed these points.


## 2. What we learned about the existing models

Before building anything we measured GraphSAGE and QGNN **the way the service runs them**:
each 50k-transaction batch from the backend is its own graph.

- **QGNN port is correct.** Re-running the notebook's own test evaluation with the saved weights
  reproduced AUPRC 0.7098 / AUROC 0.9712 (notebook: 0.7096 / 0.9713).
- **Both models lose most of their accuracy when served on batches.** They were trained on one
  large graph with the training history as context. Served on batches, on the notebook's
  enriched test sample (2.9% laundering): GraphSAGE AUPRC 0.540, QGNN 0.454 (QGNN's notebook
  number was 0.71).
- **Their thresholds don't transfer.** GraphSAGE's threshold (0.977) flagged 46k of 208k
  transactions in that test; at the real laundering rate it flags almost half of everything.
- **Bug in QGNN's notebook output:** `ibm_hi_medium_test.csv`'s `risk_score` column is attached
  to the wrong rows (an unstable sort on duplicate timestamps), which makes that file look like
  a random model (AUROC 0.49).

**Lesson that shaped everything below: train the model on the same kind of graph it will be
served on.**


## 3. Data

- **Source:** IBM AML **HI-Medium** (`HI-Medium_Trans.csv`, 31.9M transactions, 3 GB), loaded with
  `kagglehub.dataset_download(..., path="HI-Medium_Trans.csv")`. It doesn't fit in memory on this
  machine (7.6 GB RAM) with pandas, so it's read with Polars in streaming passes.
- **Window:** 1-16 Sep 2022 (31.89M rows). Rows after that are almost all laundering, an artifact
  of the dataset, so they're dropped.
- **Batches:** the window sorted by time and cut into 637 batches of 50,000 transactions, which is
  exactly what the backend poller sends.
- **Time split**, the same cut-offs GraphSAGE and QGNN were trained with, so the test period is
  unseen by every model:

| split | period | batches available | used (v1) | used (v2) |
|---|---|---|---|---|
| train | up to 2022-09-09 20:41 | 382 | 80 | 200 |
| val | to 2022-09-14 05:42 | 127 | 20 | same 20 |
| test | after | 126 | 20 | same 20 |

- **Real laundering rate** (~0.1%), not an enriched sample: test = 1,000,000 transactions,
  1,066 laundering (0.107%). A random model scores AUPRC ~0.001 here.


## 4. v1 design

**Features (computed per batch graph, from that batch only, identical at train and serve time)**

- Edge (17): log amount paid / received (converted to AUD), log received/paid ratio,
  cross-currency, cross-bank, self-transfer, hour sin/cos, weekend, night,
  payment format **one-hot** (7).
- Node (8): the notebook's account aggregates: log total sent / received, out / in count,
  unique counterparties out / in, average sent / received.
- Dropped on purpose: `hours_from_start`. The existing models use it, but it keeps growing over
  time, so new data always looks unlike training data.
- Scaling: mean / std from the training graphs, stored as tensors in the bundle (no pickled
  scikit-learn objects, so no library-version warnings).

**Model**

- Node input layer -> 2 x **two-way GINE layers** -> MLP on `[sender, receiver, edge features]`.
- GINE layers mix each transaction's features into the message (the notebook's layers didn't).
- **Two-way:** separate layers for messages along the money flow (receiver hears from sender)
  and against it (sender hears from receiver), so paying and being paid are learned separately.
- Residual connections + BatchNorm, dropout 0.2, hidden size 64 (66k parameters).

**Training**

- One batch graph per step, Adam lr 1e-3, weight decay 1e-5, gradient clipping 1.0.
- Weighted BCE with `pos_weight = sqrt(negatives / positives)` (~34). The full ratio (~1000)
  makes training unstable.
- Early stopping on validation AUPRC (patience 6); decision threshold = max F1 on validation.

**Training curve:** validation AUPRC 0.007 -> 0.32 -> 0.39 in 3 epochs, best **0.440** at
epoch 9, stopped at 15.


## 5. v1 results

Test: 20 unseen batches = 1,000,000 transactions, 1,066 laundering (0.107%). Every model scores
each 50k batch as its own graph, exactly as in production.

| model | AUPRC | AUROC | precision | recall | F1 | flagged |
|---|---|---|---|---|---|---|
| **amlgnn v1** | **0.376** | **0.979** | **31.3%** | 45.6% | **0.371** | 1,551 |
| graphsage | 0.015 | 0.858 | 0.2% | 90.7% | 0.004 | 474,995 |
| qgnn | 0.008 | 0.884 | 1.0% | 79.1% | 0.021 | 79,977 |

- **AUPRC 0.376 vs 0.015 / 0.008.** For comparison, the source notebook reached 0.038 on HI-Small.
- At its validation-chosen threshold, v1 flags 1,551 of 1M transactions; about 1 in 3 flags is
  real laundering, and it catches about half of all laundering.
- GraphSAGE and QGNN flag 47% and 8% of everything at the real laundering rate, which makes
  them unusable as alert generators without re-tuning.
- v1 scores 1M transactions in ~6s (QGNN: ~8 minutes).


## 6. v2: what was changed to improve it, and why

1. **Pattern features (6 new edge features).** IBM laundering follows known shapes: fan-out,
   fan-in, cycles, scatter-gather and bursts of transfers. Per transaction, within its batch:
   - `log_pair_count`: how many times the same sender -> receiver pair appears (repeated transfers)
   - `has_reverse_flow`: money also flows receiver -> sender (2-cycles)
   - `log_amount_vs_sender_avg`, `log_amount_vs_receiver_avg`: unusual amount for that account
   - `log_minutes_since_sender_prev`, `log_minutes_since_receiver_prev`: bursts of activity
2. **Edge updates.** Each layer also refreshes a hidden state for every transaction from its two
   accounts' new states, so the second hop passes richer messages. Updating edges as well as
   nodes has given large gains on these IBM datasets in published work (Multi-GNN / MEGA-GNN).
3. **2.5x more training data:** 200 training batches (10M transactions) instead of 80.
4. **Streaming graphs:** each batch graph is built once, saved to `data/graphs_v2/`, and loaded
   one at a time, so training on more batches fits in memory.
5. **Same validation and test batches as v1**, so the numbers compare directly.
6. Faster learning-rate decay (patience 2) and early stopping (patience 5), since validation
   AUPRC plateaued after ~9 epochs in v1.
7. **Checkpoint every epoch + resume.** The first v2 run stalled after 4 epochs (the computer went
   to sleep) and was killed with nothing saved. Training now saves
   `data/amlgnn_v2_checkpoint.pt` after every epoch and a rerun carries on from there; on Windows
   the script also asks the OS not to sleep while it runs.

Old bundles keep working: `amlgnn.py` picks each bundle's features by name from its config, and
edge updates are off unless the bundle says otherwise.


## 7. v2 results

Same 1M test transactions. Best validation AUPRC 0.530 (v1: 0.440), threshold 0.7109.

| model | AUPRC | AUROC | precision | recall | F1 | flagged |
|---|---|---|---|---|---|---|
| **amlgnn v2** | 0.452 | 0.984 | 57.4% | 41.9% | 0.485 | 779 |
| amlgnn v1 | 0.376 | 0.979 | 31.3% | 45.6% | 0.371 | 1,551 |
| graphsage | 0.015 | 0.858 | 0.2% | 90.7% | 0.004 | 474,995 |
| qgnn (from the v1 run) | 0.008 | 0.884 | 1.1% | 79.1% | 0.021 | 79,977 |

**What improved (v1 -> v2):**

- AUPRC 0.376 -> **0.452** (+20%); F1 0.371 -> **0.485** (+31%).
- Precision 31% -> **57%**: more than half of v2's alerts are real laundering, while it flags half as many transactions (779 vs 1,551). Recall dipped slightly (46% -> 42%) because its validation-chosen threshold is stricter; a lower threshold trades precision back for recall.
- Training curve: validation AUPRC 0.446 after one epoch (v1 needed 9 epochs for 0.440), best **0.531** at epoch 16, early stop at 21; ~4.3 minutes per epoch on 200 batches.
- Scores 1M transactions in ~8s.

The changes were made together (pattern features + edge updates + 2.5x data), so this run doesn't say how much each one contributed. An ablation (train with each change switched off) would answer that.

**In the service:** `amlgnn` now serves v2; v1 stays available as `amlgnn-v1`.


## 8. Ideas to improve further

Roughly in order of expected value for effort:

1. **Use all 382 training batches** (19M transactions). v2 streams graphs from disk, so this is
   only a constant change; it costs training time (~9 min per epoch on this CPU).
2. **Larger validation set.** 20 batches hold ~1,400 laundering transactions; validation AUPRC
   jumps around by +-0.05 between epochs, which makes early stopping and threshold choice noisy.
3. **Hyperparameter search** (hidden size, layers, dropout, lr, pos_weight) with Optuna, which the
   QGNN notebook already used.
4. **Ego IDs and port numbering** from Multi-GNN (Egressy et al., 2024): mark the transaction's own
   two accounts, and number each account's counterparties, so the GNN can detect cycles and
   repeated counterparties precisely. Reported large F1 gains on these IBM datasets.
5. **Longer context than one batch.** A 50k batch covers only ~35 minutes of HI-Medium traffic.
   Carry account aggregates across batches (e.g. a rolling store of each account's recent
   activity) and feed them as extra node features.
6. **Threshold by alert budget.** Pick the threshold that gives the number of alerts analysts can
   review per day, instead of max F1.
7. **Ensemble with a tabular model** (XGBoost on the same features); the source notebook found
   the two catch different cases.
8. **Check generalisation** on HI-Small / LI-Medium (lower laundering rate) before trusting it on
   other data.


## 9. Inspect the saved bundles

Runnable cells: they read the trained bundles' configs and stored test results, and score one test
batch locally (needs `data/hi_medium_batches.parquet` from a training run).


In [1]:
import pandas as pd
import torch
from pathlib import Path

rows = []
for path in sorted(Path("model").glob("amlgnn_v*.pt")):
    cfg = torch.load(path, map_location="cpu", weights_only=False)["config"]
    rows.append({
        "bundle": path.name,
        "edge features": len(cfg["edge_features"]),
        "edge updates": cfg.get("edge_updates", False),
        "val AUPRC": round(cfg["val_auprc"], 4),
        "threshold": round(cfg["decision_threshold"], 4),
        **{f"test {k}": round(v, 4) if isinstance(v, float) else v for k, v in cfg["test_metrics"].items()},
    })
pd.DataFrame(rows)

,bundle,edge features,edge updates,val AUPRC,threshold,test auprc,test auroc,test precision,test recall,test f1,test flagged
0,amlgnn_v1.pt,17,False,0.4398,0.5085,0.3761,0.9787,0.3133,0.4559,0.3714,1551
1,amlgnn_v2.pt,23,True,0.5305,0.7109,0.4523,0.9836,0.5738,0.4193,0.4846,779


In [2]:
from amlgnn import ALL_EDGE_FEATURES, NODE_FEATURES, PATTERN_FEATURES

print("node features:", NODE_FEATURES)
print("edge features:", ALL_EDGE_FEATURES)
print("added in v2:  ", PATTERN_FEATURES)

node features: ['log_total_sent', 'log_total_received', 'log_tx_count_out', 'log_tx_count_in', 'log_unique_counterparties_out', 'log_unique_counterparties_in', 'log_avg_sent', 'log_avg_received']
edge features: ['log_amount_paid', 'log_amount_received', 'log_amount_ratio', 'is_cross_currency', 'is_cross_bank', 'is_self_transfer', 'hour_sin', 'hour_cos', 'is_weekend', 'is_night', 'format_ACH', 'format_Bitcoin', 'format_Cash', 'format_Cheque', 'format_Credit Card', 'format_Reinvestment', 'format_Wire', 'log_pair_count', 'has_reverse_flow', 'log_amount_vs_sender_avg', 'log_amount_vs_receiver_avg', 'log_minutes_since_sender_prev', 'log_minutes_since_receiver_prev']
added in v2:   ['log_pair_count', 'has_reverse_flow', 'log_amount_vs_sender_avg', 'log_amount_vs_receiver_avg', 'log_minutes_since_sender_prev', 'log_minutes_since_receiver_prev']


In [3]:
import polars as pl
from amlgnn import MODEL_DIR, infer_amlgnn, load_amlgnn_bundle

bundle = load_amlgnn_bundle()  # v2; pass MODEL_DIR / "amlgnn_v1.pt" for v1
test = pl.read_parquet("data/hi_medium_batches.parquet").filter(pl.col("split") == "test")
batch = test.filter(pl.col("batch") == test["batch"].min()).sort("pos").drop("split", "batch", "pos")

result = infer_amlgnn(batch, bundle)
scored = result["frame"].select("sender_id", "receiver_id", "amount", "Payment Format", "Is Laundering", "risk_score")
print(f"{result['n_edges']:,} transactions, {result['n_nodes']:,} accounts, {sum(result['flagged'])} flagged")
scored.sort("risk_score", descending=True).head(10)

50,000 transactions, 56,166 accounts, 31 flagged


sender_id,receiver_id,amount,Payment Format,Is Laundering,risk_score
str,str,f64,str,i8,f64
"""81D8D2940_0048741""","""815C682E0_00741""",6009.892,"""ACH""",1,0.97858
"""824C0A830_0295233""","""8234B06D0_0295764""",4155.88,"""ACH""",1,0.97412
"""80039F770_01208""","""8006DF2D0_014""",15489.2096,"""ACH""",1,0.971409
"""824747870_0298720""","""815E78380_004""",12557.1873,"""ACH""",1,0.970388
"""83ABCEF60_0283264""","""83ABCFB90_02131359""",3182.634,"""ACH""",1,0.969619
"""817FAC900_0264284""","""8240B8360_028219""",1.4952e7,"""ACH""",1,0.969316
"""80BA0DB10_0131226""","""80E7EBF80_013""",20832.722988,"""ACH""",0,0.968003
"""80DDC5960_006422""","""81D397290_0177210""",15193.3538,"""ACH""",1,0.964097
"""80079F730_02954""","""808F8BC10_0018140""",8470.425688,"""ACH""",1,0.959128
